Hey! Please follow the next steps in order to run the full EEG analysis pipeline. 
I hope you have fun with it :-)

Sophie

# General instructions
- Give a check to the glossary below to make sure we are on the same page with the terminology / variable names.
- Set the following **global parameters** depending on what mode you want to run the pipeline with.
    - _test_
    - _show_
    - _load_
    - _save_
- Write in the _pids_ list the participants ID (as strings) for which you want to run the pipeline for
- Simply run all the other cells (in the right order!)

## Glossary
- "raw" = as in MNE, raw is used to refer to **continuous**, i.e. "non-epoched" EEG recordings (and not "uncleaned" ones!)
- _pid_ = Participant ID
- _cid_ = Condition ID
- _test_ = whether you want to run in testing mode or not; testing mode runs faster by processing only subsets of the input (e.g. only one participant instead of all, only one type of epoch instead of all, ...). This is typically to use to make sure everything runs smoothly, for example before saving anything.
- _show_ = whether you want to show in the notebook the generated figures or not.
- _load_ = whether you want to load data from existing files (when possible) or regenerate them from scratch.
- _save_ = whether you want to export results (recordings, plots, ...) or not.


In [ ]:
# Set global params
test = False
show = False
load = False
save = False

In [ ]:
# Set IDs of participants you want to run the pipeline for
pids = []

In [ ]:
# Import required packages
from utils.gen_utils import get_pid_cids
from preprocessing.behavior_to_eeg import get_times_retrieval_phases, get_trace_df, get_retrieval_df, extract_behav_events, define_eeg_epochs 
from visualization.vis_eeg import plot_schematic_epo_def, plot_evk_by_grp, plot_psd_avg_by_grp, compare_epo_psd, compare_band_metric, compare_found_peaks
from preprocessing.extract_eeg  import get_raw_to_epoch, get_all_epo_objects
from processing.process_eeg import compute_psd_by_key, get_psd_df, get_psd_avg_df, get_band_metrics_df
import numpy as np

%load_ext autoreload
%autoreload 2

# 0. Data cleaning
For this step you need to use code adapted from Stav's pipeline (_eeg_pipeline_ shared repository, on **_Sophie_branch_ (!)** git branch).

# 1. Epoching
In this step, the continuous, clean EEG data will be epoched based on the participant's behavior. There will be different types of epoched data, kept separate, for example when participants where still (_Static_ epoch) or initiated movement (_MovOn_ epoch).

### 1.1 Define events based on behavior

In [ ]:
for pid in pids:
    
    # Extract behavioral events
    block_times = get_times_retrieval_phases(pid)
    df_trace = get_trace_df(pid)
    retrieval_df = get_retrieval_df(pid)
    behav_events_df = extract_behav_events(pid, block_times, retrieval_df, df_trace, save=save)
    
    ## Define EEG epochs based on behavioral events
    eeg_epochs_df = define_eeg_epochs(behav_events_df, pid, save=save)
    
    # Plot the behavior and relative epochs extracted of each trial
    plot_schematic_epo_def(behav_events_df, eeg_epochs_df, show=show, save=save, pid=pid)

### 1.2 Extract EEG epochs based on defined events

In [ ]:
for pid in pids:
    blocks = get_pid_cids(pid, test=test)
    for block in blocks:
        
        kwargs = dict(pid=pid, cid=block)
        
        # Get raw (continuous) clean data
        raw_rec = get_raw_to_epoch(**kwargs)
        epos_dict = get_all_epo_objects(raw_rec, save=save, load=load, verbose=False, test=test, **kwargs)
        
        # Visualize some results of the epoching
        
        # Plot evoked response 
        _ = plot_evk_by_grp(epos_dict, show=show, save=save, **kwargs)
        
        # Plot PSD
        psd_by_rec = compute_psd_by_key(epos_dict)
        _ = plot_psd_avg_by_grp(psd_by_rec, show=show, save=save, **kwargs)

# 2. Compute spectral features
At this point we are ready to compute spectral features (power spectra, oscillations) of interest. Computations are done separately for each epoch-type and block number (until we're blinded, then block number will be replaced by stimulation condition), that we will compare to each other. To facilitate comparisons/analyses, all features are stored in tables.
- raw (continu

## 2.1 Full PSD tables
Compute power spectra for each block and epoch type, both in logarithmic scale (better for plotting) as well as in linear scale (needed to later compute averages). Power spectra are computed for each epoch and channel separately, then averaged across epochs and then channels of the same epoch-type in the same block.
- Note: In these tables, pw_avg and pw_std refer to the power average/std across channels.

In [ ]:
psd_df_log = get_psd_df(load=load, test=test, save=save, log=True)
psd_df_lin = get_psd_df(load=load, test=test, save=save, log=False)

## 2.2 Average PSD tables
Compute average PSDs, from full linear (!) PSDs. They are computed by epoch-type and block, first by averaging across different blocks of the same participant, then averaging participants together.
- Note: In these tables, pw_avg and pw_std refer to the power average/std across participants.

In [ ]:
psd_avg_df_log = get_psd_avg_df(load=load, save=save, log=True)
psd_avg_df_lin = get_psd_avg_df(load=load, save=save, log=False)

## 2.3 Oscillatory features tables
Compute features of oscillations (absolute/relative band power, SNR, FOOOF peaks, ...) from the full linear power spectra of each participant, block and epoch type.
- Note: In these tables, pw_avg and pw_std refer to the power average/std across channels.

In [ ]:
osc_df = get_band_metrics_df(load=load, test=test, save=save)

# 3. Display all spectral features

In [ ]:
plot_kwargs = dict(show=show, save=save)

## 3.1 Across participants

In [ ]:
# Power spectra
compare_epo_psd(psd_avg_df_log, 'epo_type', plot_subj='average', **plot_kwargs)

In [ ]:
# Power in canonical bands
compare_band_metric(osc_df, super_col='epo_type', metric_name='abs_pw', **plot_kwargs)
compare_band_metric(osc_df, super_col='epo_type', metric_name='rel_pw', **plot_kwargs)

In [ ]:
# SNR 
compare_band_metric(osc_df, super_col='epo_type', metric_name='osc_snr', **plot_kwargs)

In [ ]:
# FOOOF peaks
compare_found_peaks(osc_df, metric_name='pk', **plot_kwargs)

## 3.2 For each participant separately  


In [ ]:
# Power spectra
for pid in pids:
    df = psd_df_lin[psd_df_lin['pid'] == pid]
    df['pw_avg'] = np.log10(df['pw_avg'])
    plot_df = df.groupby(['cond', 'epo_type', 'freq'], as_index=False).agg(  # average together blocks of the same condition
            pid=("pid", 'first'),
            freq=("freq", 'first'),
            pw_avg=('pw_avg', 'mean'),
            pw_std=('pw_avg', 'std'),
            N=('pw_avg', 'count'),
        )
    compare_epo_psd(plot_df, 'epo_type', plot_subj=pid, **plot_kwargs)  # sem_n is number of blocks by condition

In [ ]:
# Power in canonical bands
for pid in pids:
    df = osc_df[osc_df['pid'] == pid]
    compare_band_metric(df, super_col='epo_type', metric_name='abs_pw', **plot_kwargs)
    compare_band_metric(df, super_col='epo_type', metric_name='rel_pw', **plot_kwargs)

In [ ]:
# SNR
for pid in pids:
    df = osc_df[osc_df['pid'] == pid]
    compare_band_metric(df, super_col='epo_type', metric_name='osc_snr', **plot_kwargs)